In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go
import pickle

df = pd.read_csv('../data/processed/dataset_final.csv')
print(f"Dataset: {df.shape[0]} filas | {df.shape[1]} columnas")
print("Listo")

Dataset: 3676 filas | 30 columnas
Listo


In [3]:
# Columnas que usa el modelo para predecir
features = [
    'anos_experiencia',
    'seniority',
    'modalidad', 
    'rol_estandar',
    'provincia'
]
target = 'salario_usd_mensual'

# Trabajamos con una copia limpia
df_modelo = df[features + [target]].copy()

# Eliminar nulos
df_modelo = df_modelo.dropna()
print(f"Filas para el modelo: {len(df_modelo)}")

# Convertir años de experiencia a numérico
df_modelo['anos_experiencia'] = pd.to_numeric(df_modelo['anos_experiencia'], errors='coerce')
df_modelo = df_modelo.dropna()
print(f"Filas después de limpiar: {len(df_modelo)}")

print("\nDistribución de features categóricos:")
for col in ['seniority', 'modalidad']:
    print(f"\n{col}:")
    print(df_modelo[col].value_counts())

Filas para el modelo: 3676
Filas después de limpiar: 3676

Distribución de features categóricos:

seniority:
seniority
senior         2029
semi-senior    1198
junior          449
Name: count, dtype: int64

modalidad:
modalidad
100% remoto                      1811
híbrido (presencial y remoto)    1586
100% presencial                   279
Name: count, dtype: int64


In [5]:
df_encoded = df_modelo.copy()

# Label Encoding para cada columna categórica
encoders = {}
cols_categoricas = ['seniority', 'modalidad', 'rol_estandar', 'provincia']

for col in cols_categoricas:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    encoders[col] = le
    print(f"{col}: {len(le.classes_)} categorías únicas")

print(f"\nShape final: {df_encoded.shape}")
df_encoded.head(3)

seniority: 3 categorías únicas
modalidad: 3 categorías únicas
rol_estandar: 14 categorías únicas
provincia: 24 categorías únicas

Shape final: (3676, 6)


,anos_experiencia,seniority,modalidad,rol_estandar,provincia,salario_usd_mensual
0,3,1,2,9,4,1871.220566
1,10,2,1,10,0,1594.634873
2,5,1,1,10,4,2235.469449


In [6]:
### Celda 4 — Split y entrenamiento del modelo
X = df_encoded[features]
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} filas | Test: {len(X_test)} filas")

modelo = RandomForestRegressor(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

print("Modelo entrenado")

Train: 2940 filas | Test: 736 filas
Modelo entrenado


In [7]:
### Celda 5 — Evaluación del modelo
y_pred = modelo.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== MÉTRICAS DEL MODELO ===")
print(f"MAE (Error promedio):  USD {mae:,.0f}/mes")
print(f"R² (Precisión):        {r2:.2f}")
print(f"\nInterpretación:")
print(f"El modelo se equivoca en promedio USD {mae:,.0f} por mes")
print(f"Explica el {r2*100:.1f}% de la variación salarial")

=== MÉTRICAS DEL MODELO ===
MAE (Error promedio):  USD 886/mes
R² (Precisión):        0.19

Interpretación:
El modelo se equivoca en promedio USD 886 por mes
Explica el 19.5% de la variación salarial


In [8]:
### Celda 6 — Importancia de variables
importancias = pd.DataFrame({
    'feature': features,
    'importancia': modelo.feature_importances_
}).sort_values('importancia', ascending=True)

fig = px.bar(
    importancias,
    x='importancia',
    y='feature',
    orientation='h',
    title='Importancia de Variables en la Predicción del Salario',
    labels={'importancia': 'Importancia', 'feature': ''},
    color='importancia',
    color_continuous_scale='Blues'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

In [9]:
### Celda 7 — Guardar el modelo
import pickle

with open('../data/processed/modelo_salario.pkl', 'wb') as f:
    pickle.dump(modelo, f)

with open('../data/processed/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

print("Modelo y encoders guardados")

Modelo y encoders guardados


In [10]:
### Celda 8 — Modelo mejorado con más features
features_v2 = [
    'anos_experiencia',
    'seniority',
    'modalidad',
    'rol_estandar',
    'provincia',
    'sueldo_dolarizado',
    'tamano_empresa'
]

df_modelo_v2 = df[features_v2 + [target]].copy()
df_modelo_v2 = df_modelo_v2.dropna()

# Convertir numéricos
df_modelo_v2['anos_experiencia'] = pd.to_numeric(df_modelo_v2['anos_experiencia'], errors='coerce')
df_modelo_v2 = df_modelo_v2.dropna()

# Encoding
df_encoded_v2 = df_modelo_v2.copy()
encoders_v2 = {}
cols_cat_v2 = ['seniority', 'modalidad', 'rol_estandar', 'provincia', 
               'sueldo_dolarizado', 'tamano_empresa']

for col in cols_cat_v2:
    le = LabelEncoder()
    df_encoded_v2[col] = le.fit_transform(df_encoded_v2[col].astype(str))
    encoders_v2[col] = le

# Split y entrenamiento
X2 = df_encoded_v2[features_v2]
y2 = df_encoded_v2[target]

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

modelo_v2 = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
modelo_v2.fit(X_train2, y_train2)

y_pred2 = modelo_v2.predict(X_test2)
mae2 = mean_absolute_error(y_test2, y_pred2)
r2_v2 = r2_score(y_test2, y_pred2)

print("=== MODELO V2 — MÁS FEATURES ===")
print(f"MAE:  USD {mae2:,.0f}/mes")
print(f"R²:   {r2_v2:.2f} ({r2_v2*100:.1f}%)")
print(f"\nMejora en R²: {(r2_v2 - r2)*100:.1f} puntos porcentuales")

=== MODELO V2 — MÁS FEATURES ===
MAE:  USD 823/mes
R²:   0.32 (32.3%)

Mejora en R²: 12.8 puntos porcentuales


In [11]:
### Celda 10 — Guardar modelo v2
with open('../data/processed/modelo_salario.pkl', 'wb') as f:
    pickle.dump(modelo_v2, f)

with open('../data/processed/encoders.pkl', 'wb') as f:
    pickle.dump(encoders_v2, f)

# Guardar también la lista de features para la app
import json
with open('../data/processed/features.json', 'w') as f:
    json.dump(features_v2, f)

print("Modelo v2 guardado")
print(f"Features: {features_v2}")

Modelo v2 guardado
Features: ['anos_experiencia', 'seniority', 'modalidad', 'rol_estandar', 'provincia', 'sueldo_dolarizado', 'tamano_empresa']


In [12]:
### Celda 11 — Preparar datos para clustering
features_cluster = [
    'anos_experiencia',
    'salario_usd_mensual',
    'seniority',
    'modalidad',
    'rol_estandar'
]

df_cluster = df_encoded_v2[['anos_experiencia', 'salario_usd_mensual']].copy()
df_cluster['seniority'] = df_encoded_v2['seniority']
df_cluster['modalidad'] = df_encoded_v2['modalidad']
df_cluster['rol_estandar'] = df_encoded_v2['rol_estandar']
df_cluster = df_cluster.dropna()

# Escalar para que todas las variables tengan el mismo peso
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster)

print(f"Filas para clustering: {len(df_cluster)}")
print("Datos escalados")

Filas para clustering: 3676
Datos escalados


In [13]:
### Celda 12 — Elegir número óptimo de clusters (método del codo)
inercias = []
K = range(2, 10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inercias.append(kmeans.inertia_)

fig = px.line(
    x=list(K),
    y=inercias,
    title='Método del Codo — Número óptimo de clusters',
    labels={'x': 'Número de clusters (K)', 'y': 'Inercia'},
    markers=True
)
fig.update_traces(line_color='#2563eb')
fig.show()

In [14]:
### Celda 13 — Aplicar KMeans con K=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(df_scaled)

# Agregar cluster al dataset original
df['cluster'] = df_cluster['cluster']

# Perfil de cada cluster
perfil = df.groupby('cluster').agg(
    cantidad=('salario_usd_mensual', 'count'),
    salario_mediano_usd=('salario_usd_mensual', 'median'),
    experiencia_mediana=('anos_experiencia', 'median'),
    seniority_top=('seniority', lambda x: x.mode()[0]),
    modalidad_top=('modalidad', lambda x: x.mode()[0]),
    rol_top=('rol_estandar', lambda x: x.mode()[0])
).round(1)

print("=== PERFIL DE CADA CLUSTER ===")
print(perfil.to_string())

=== PERFIL DE CADA CLUSTER ===
         cantidad  salario_mediano_usd  experiencia_mediana seniority_top                  modalidad_top                    rol_top
cluster                                                                                                                            
0            1376               1542.5                  3.0   semi-senior                    100% remoto                  Developer
1            1016               2661.9                 13.0        senior                    100% remoto                  Developer
2             748               2811.0                 14.0        senior  híbrido (presencial y remoto)        Engineering manager
3             536               2318.0                  6.5        senior  híbrido (presencial y remoto)  Data analyst or scientist


In [15]:
### Celda 14 — Nombrar los clusters
nombres_cluster = {
    0: 'Junior Remoto',
    1: 'Senior Remoto',
    2: 'Senior Manager Híbrido',
    3: 'Senior Data Híbrido'
}

df['cluster_nombre'] = df['cluster'].map(nombres_cluster)

print("=== SEGMENTOS DEL MERCADO TECH ARGENTINO ===")
print(f"\nCluster 0 — Junior Remoto (n={1376})")
print(f"   Salario mediano: USD 1.543/mes | Experiencia: 3 años")
print(f"   Perfil: Developer semi-senior trabajando 100% remoto")

print(f"\nCluster 1 — Senior Remoto (n={1016})")
print(f"   Salario mediano: USD 2.662/mes | Experiencia: 13 años")
print(f"   Perfil: Developer senior 100% remoto — el más común en tech")

print(f"\nCluster 2 — Senior Manager Híbrido (n={748})")
print(f"   Salario mediano: USD 2.811/mes | Experiencia: 14 años")
print(f"   Perfil: Engineering manager senior en modalidad híbrida")

print(f"\nCluster 3 — Senior Data Híbrido (n={536})")
print(f"   Salario mediano: USD 2.318/mes | Experiencia: 6.5 años")
print(f"   Perfil: Data analyst/scientist senior en modalidad híbrida")

=== SEGMENTOS DEL MERCADO TECH ARGENTINO ===

Cluster 0 — Junior Remoto (n=1376)
   Salario mediano: USD 1.543/mes | Experiencia: 3 años
   Perfil: Developer semi-senior trabajando 100% remoto

Cluster 1 — Senior Remoto (n=1016)
   Salario mediano: USD 2.662/mes | Experiencia: 13 años
   Perfil: Developer senior 100% remoto — el más común en tech

Cluster 2 — Senior Manager Híbrido (n=748)
   Salario mediano: USD 2.811/mes | Experiencia: 14 años
   Perfil: Engineering manager senior en modalidad híbrida

Cluster 3 — Senior Data Híbrido (n=536)
   Salario mediano: USD 2.318/mes | Experiencia: 6.5 años
   Perfil: Data analyst/scientist senior en modalidad híbrida


In [16]:
### Celda 15 — Visualizar clusters
fig = px.scatter(
    df.dropna(subset=['anos_experiencia', 'salario_usd_mensual', 'cluster_nombre']),
    x='anos_experiencia',
    y='salario_usd_mensual',
    color='cluster_nombre',
    title='Segmentos de Profesionales Tech en Argentina',
    labels={
        'anos_experiencia': 'Años de experiencia',
        'salario_usd_mensual': 'Salario USD/mes',
        'cluster_nombre': 'Segmento'
    },
    color_discrete_sequence=['#93c5fd', '#2563eb', '#1e3a8a', '#60a5fa'],
    opacity=0.6
)
fig.show()

In [17]:
### Celda 16 — Guardar dataset final con clusters
df.to_csv('../data/processed/dataset_final.csv', index=False)

with open('../data/processed/modelo_kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Dataset final actualizado con clusters")
print("Modelos guardados:")
print("   - modelo_salario.pkl  → RandomForest")
print("   - modelo_kmeans.pkl   → KMeans")
print("   - scaler.pkl          → StandardScaler")
print("   - encoders.pkl        → LabelEncoders")
print("   - features.json       → Lista de features")

Dataset final actualizado con clusters
Modelos guardados:
   - modelo_salario.pkl  → RandomForest
   - modelo_kmeans.pkl   → KMeans
   - scaler.pkl          → StandardScaler
   - encoders.pkl        → LabelEncoders
   - features.json       → Lista de features
